# Chapter 6: Bayesian formalized

Heavy PyMC sampling is done by `generate.py` and saved to `data/`. This notebook does the closed-form Beta math live so it runs in seconds. To reload the saved PyMC traces, see the inline cell at the bottom.

In [1]:
import numpy as np
from scipy.stats import beta as beta_dist
from expkit.sim.coin import bernoulli_sequence
from expkit.inference.bayes import bayes_factor_point_vs_uniform, conjugate_posterior_predictive
from expkit.io.samples import load_idata
from expkit.plot.style import apply_style
apply_style()

## Loop B: priors that argue

In [2]:
for k, n in [(6, 10), (600, 1000)]:
    print(f'{k}/{n}:')
    for name, a0, b0 in [('flat', 1, 1), ('skeptical', 50, 50), ('expects-tails', 2, 8), ('expects-heads', 8, 2)]:
        post_a, post_b = a0 + k, b0 + (n - k)
        mean = post_a / (post_a + post_b)
        lo = beta_dist.ppf(0.025, post_a, post_b); hi = beta_dist.ppf(0.975, post_a, post_b)
        print(f'    {name:<14} prior -> Beta({post_a:.0f}, {post_b:.0f})  mean={mean:.3f}  95% CI=[{lo:.3f}, {hi:.3f}]')

6/10:
    flat           prior -> Beta(7, 5)  mean=0.583  95% CI=[0.308, 0.833]
    skeptical      prior -> Beta(56, 54)  mean=0.509  95% CI=[0.416, 0.602]
    expects-tails  prior -> Beta(8, 12)  mean=0.400  95% CI=[0.203, 0.616]
    expects-heads  prior -> Beta(14, 6)  mean=0.700  95% CI=[0.488, 0.874]
600/1000:
    flat           prior -> Beta(601, 401)  mean=0.600  95% CI=[0.569, 0.630]
    skeptical      prior -> Beta(650, 450)  mean=0.591  95% CI=[0.562, 0.620]
    expects-tails  prior -> Beta(602, 408)  mean=0.596  95% CI=[0.566, 0.626]
    expects-heads  prior -> Beta(608, 402)  mean=0.602  95% CI=[0.572, 0.632]


## Loop D: Bayes factor accumulates evidence

In [3]:
rng = np.random.default_rng(0)
biased = rng.binomial(1, 0.55, size=2000)
for n in [50, 100, 500, 1000, 2000]:
    bf = bayes_factor_point_vs_uniform(biased[:n], point=0.5)
    print(f'biased data, N={n:>5}: BF_10 = {bf:.2f}')

biased data, N=   50: BF_10 = 0.18
biased data, N=  100: BF_10 = 0.13
biased data, N=  500: BF_10 = 0.06
biased data, N= 1000: BF_10 = 0.06
biased data, N= 2000: BF_10 = 77.67


## Loop E: posterior predictive

In [4]:
seq = bernoulli_sequence(50, p=0.6, seed=11)
preds = conjugate_posterior_predictive(seq, new_n=20, n_samples=10000, seed=0)
print('predictive distribution over heads in next 20 tosses:')
for q in [0.025, 0.25, 0.5, 0.75, 0.975]:
    print(f'    quantile {q}: {np.quantile(preds, q):.1f}')

predictive distribution over heads in next 20 tosses:
    quantile 0.025: 8.0
    quantile 0.25: 11.0
    quantile 0.5: 13.0
    quantile 0.75: 15.0
    quantile 0.975: 18.0


## Reload PyMC trace from disk

The PyMC posterior produced by `generate.py` is saved as a netCDF arviz `InferenceData` object. We can reload it without running the sampler again.

In [5]:
from pathlib import Path
trace_path = Path('data/posterior_chapter6.nc')
if trace_path.exists():
    idata = load_idata(trace_path)
    p = idata.posterior['p'].values.ravel()
    print(f'PyMC posterior: mean={p.mean():.4f}  95% CI=[{np.quantile(p, 0.025):.4f}, {np.quantile(p, 0.975):.4f}]')
else:
    print('Run generate.py to produce the trace.')

Run generate.py to produce the trace.
